# Exercise 5 — SafeAgent

**SafeAgent** wraps `safe_ask` with persistent history and a `reset_budget()` method for multi-turn sessions.  It takes the same constructor arguments as `safe_ask` and stores a `BudgetTracker` that persists across calls — so the budget caps the total calls in a session, not just per invocation.

In [ ]:
import json
def validate_text(text, max_length=None, banned=None):
    text_str = str(text)
    if max_length is not None and len(text_str) > max_length:
        return False, ("text exceeds max_length ("
                       + str(len(text_str)) + " > " + str(max_length) + " chars)")
    if banned:
        lower = text_str.lower()
        for pattern in banned:
            if str(pattern).lower() in lower:
                return False, "banned pattern found: " + repr(pattern)
    return True, ""

class Guard:
    def __init__(self, max_length=None, banned=None):
        self.max_length = max_length
        self.banned = list(banned) if banned else []
    def check(self, text):
        return validate_text(text, self.max_length, self.banned)
class ApprovalGate:
    def __init__(self, approve_fn=None):
        self._approve_fn = approve_fn if approve_fn is not None else (lambda action: True)
    def check(self, action):
        try:
            result = bool(self._approve_fn(str(action)))
        except Exception:
            result = False
        return (True, "approved") if result else (False, "rejected by approval gate")
class BudgetTracker:
    def __init__(self, max_calls=None):
        self.max_calls = max_calls
        self._count = 0
    def ok(self):
        if self.max_calls is not None and self._count >= self.max_calls:
            return False, ("budget exceeded (" + str(self._count)
                           + "/" + str(self.max_calls) + " calls)")
        return True, ""
    def record(self): self._count += 1
    def reset(self): self._count = 0
    @property
    def count(self): return self._count
def safe_ask(query, agent_fn, input_guard=None, output_guard=None,
             budget=None, gate=None, llm_fn=None):
    record = {"query": query, "answer": None, "blocked": False, "reason": ""}
    if input_guard is not None:
        ok, reason = input_guard.check(str(query))
        if not ok:
            record["blocked"] = True; record["reason"] = "input: " + reason; return record
    if budget is not None:
        ok, reason = budget.ok()
        if not ok:
            record["blocked"] = True; record["reason"] = "budget: " + reason; return record
        budget.record()
    if gate is not None:
        approved, reason = gate.check(str(query))
        if not approved:
            record["blocked"] = True; record["reason"] = "gate: " + reason; return record
    try:
        answer = str(agent_fn(query, llm_fn=llm_fn))
    except Exception as exc:
        answer = "Error: " + str(exc)
    if output_guard is not None:
        ok, reason = output_guard.check(answer)
        if not ok:
            record["blocked"] = True; record["reason"] = "output: " + reason
            record["answer"] = "[blocked]"; return record
    record["answer"] = answer
    return record
_echo_agent = lambda query, llm_fn=None: 'Answer: ' + str(query)

# ── Exercise: implement SafeAgent ─────────────────────────────────────────────

class SafeAgent:
    """Agent wrapped in all four guardrail layers with persistent history."""

    def __init__(self, agent_fn, input_guard=None, output_guard=None,
                 budget=None, gate=None, llm_fn=None):
        # TODO: store all arguments as instance attributes
        # initialise self._history = []
        pass

    def ask(self, query):
        # TODO: call safe_ask(...) with all stored guardrails,
        # append the record to self._history, return the record
        return {}

    def history(self):
        # TODO: return a copy of self._history
        return []

    def clear_history(self):
        # TODO: clear self._history
        pass

    def reset_budget(self):
        # TODO: if self._budget is not None, call self._budget.reset()
        pass


### Checks

In [ ]:
checks = 0

# 1 — SafeAgent constructs
try:
    agent = SafeAgent(_echo_agent)
    checks += 1; print("✅ 1 SafeAgent constructs")
except Exception as e:
    print("❌ 1:", e)

# 2 — ask() calls through to agent_fn
try:
    agent = SafeAgent(_echo_agent)
    r = agent.ask("hi")
    assert r["answer"] == "Answer: hi" and not r["blocked"]
    checks += 1; print("✅ 2 ask() calls agent_fn and returns result")
except Exception as e:
    print("❌ 2:", e)

# 3 — blocked query is recorded in history
try:
    guard = Guard(max_length=5)
    agent = SafeAgent(_echo_agent, input_guard=guard)
    r = agent.ask("this is way too long")
    assert r["blocked"]
    assert len(agent.history()) == 1
    checks += 1; print("✅ 3 blocked query is recorded in history")
except Exception as e:
    print("❌ 3:", e)

# 4 — history() grows with each ask
try:
    agent = SafeAgent(_echo_agent)
    agent.ask("q1"); agent.ask("q2"); agent.ask("q3")
    assert len(agent.history()) == 3
    checks += 1; print("✅ 4 history() grows with each ask()")
except Exception as e:
    print("❌ 4:", e)

# 5 — clear_history() empties history
try:
    agent = SafeAgent(_echo_agent)
    agent.ask("q")
    agent.clear_history()
    assert agent.history() == []
    checks += 1; print("✅ 5 clear_history() empties history")
except Exception as e:
    print("❌ 5:", e)

# 6 — reset_budget() resets the tracker so calls are allowed again
try:
    b = BudgetTracker(max_calls=1)
    agent = SafeAgent(_echo_agent, budget=b)
    agent.ask("first")     # uses the budget
    r2 = agent.ask("second")   # should be blocked
    assert r2["blocked"]
    agent.reset_budget()
    r3 = agent.ask("third")    # should pass now
    assert not r3["blocked"]
    checks += 1; print("✅ 6 reset_budget() allows calls again after exhaustion")
except Exception as e:
    print("❌ 6:", e)

print(f"\n{checks}/6 checks passed!")
